# Libraries

In [ ]:
import pandas as pd
import os

# Spark library for big data processing
!pip install pyspark spark-nlp
import sparknlp
from pyspark.sql import SparkSession
from pyspark.ml.feature import NGram, HashingTF, MinHashLSH
from sparknlp.annotator import Tokenizer, Normalizer, StopWordsCleaner, LemmatizerModel
from sparknlp.base import DocumentAssembler, Finisher
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, size

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.1/697.1 kB 7.9 MB/s eta 0:00:00


# Kaggle Dataset (Books_rating)

In [ ]:
# Here I replaced my username and key with "XXXX". To run the whole code is needed to put an API's username and key.
os.environ['KAGGLE_USERNAME'] = 'XXXX'
os.environ['KAGGLE_KEY'] = 'XXXX'

In [ ]:
!kaggle datasets download -d mohamedbakhet/amazon-books-reviews

Dataset URL: https://www.kaggle.com/datasets/mohamedbakhet/amazon-books-reviews
License(s): CC0-1.0
 99% 1.05G/1.06G [00:07<00:00, 37.7MB/s]
100% 1.06G/1.06G [00:07<00:00, 143MB/s] 


In [ ]:
!unzip amazon-books-reviews.zip

Archive:  amazon-books-reviews.zip
  inflating: Books_rating.csv        
  inflating: books_data.csv          


In [ ]:
# As I am going to use the books_ratings, I will get just this dataset
books_rating = pd.read_csv('/content/Books_rating.csv')

In [ ]:
# I also copy the original variable to avoid modifying the original df
df_books_rating = books_rating

In [ ]:
df_books_rating.head(2)

,Id,Title,Price,User_id,profileName,review/helpfulness,review/score,review/time,review/summary,review/text
0,1882931173,Its Only Art If Its Well Hung!,NaN,AVCGYZL8FQQTD,"Jim of Oz ""jim-of-oz""",7/7,4.0,940636800,Nice collection of Julie Strain images,This is only for Julie Strain fans. It's a col...
1,0826414346,Dr. Seuss: American Icon,NaN,A30TK6U7DNS82R,Kevin Killian,10/10,5.0,1095724800,Really Enjoyed It,I don't care much for Dr. Seuss but after read...


In [ ]:
length_1 = len(df_books_rating)
length_1

3000000

# Data pre-processing

When working on the project, at the end I realized that there were some cases that should be considered duplicate as they contain same "Book title", "User_id" and "review/text".

For example:
Book Fahrenheit 451 (B000K0G43C) by user A20EEWWSFMZ1PN: has the same review, but differs review/helpfulness and review/time.

This is why I decided to previously remove duplicates, only considering columns "Id", "User_id", "review/text".

In [ ]:
# Remove duplicated reviews by "Id", "User_id", "review/text"
df_books_rating = df_books_rating.drop_duplicates(subset=["Title", "User_id", "review/text"])

In [ ]:
# Leaving attributes to be used
df_books_rating = df_books_rating[['Id', 'Title', 'review/text']]

In [ ]:
length_2 = len(df_books_rating)
percentage_2 = round((length_2/length_1)*100,2)
print(f"New length is: {length_2}")
print(f"Rows removed: {length_1 - length_2}")
print(f"Percentage remaining: {percentage_2}%")

New length is: 2619348
Rows removed: 380652
Percentage remaining: 87.31%


In [ ]:
# Check missing values
df_books_rating.isnull().sum()

,0
Id,0
Title,208
review/text,8


Before removing the missing values, I will print a message with the number and percentage of them, to know beforehand if it is a high size or not from the dataset being used.

In [ ]:
# Number and percentage of removed rows
total_rows = len(df_books_rating)

missing_titles = df_books_rating['Title'].isnull().sum()
percentage_titles = round((missing_titles/total_rows)*100,4)

missing_reviews = df_books_rating['review/text'].isnull().sum()
percentage_reviews = round((missing_reviews/total_rows)*100,4)

threshold_for_dropping = 0.4

print(f"Number of rows with missing value in column Title: {missing_titles}")
print(f"Percentage of rows with missing value in column Title: {percentage_titles}%")

if percentage_titles > threshold_for_dropping:
    print("BE CAREFUL! More than 40% of the rows have missing values in the attribute Title")
else:
    print("You can delete those rows!")

print("--------------------------------------------------------------------")

print(f"Number of rows with missing value in column review/text: {missing_reviews}")
print(f"Percentage of rows with missing value in column review/text: {percentage_reviews}%")

if percentage_reviews > threshold_for_dropping:
    print("BE CAREFUL! More than 40% of the rows have missing values in the attribute review/text")
else:
    print("You can delete those rows!")

Number of rows with missing value in column Title: 208
Percentage of rows with missing value in column Title: 0.0079%
You can delete those rows!
--------------------------------------------------------------------
Number of rows with missing value in column review/text: 8
Percentage of rows with missing value in column review/text: 0.0003%
You can delete those rows!


In [ ]:
# Remove missing values
df_books_rating = df_books_rating.dropna(subset = ['Title', 'review/text'])

In [ ]:
# Check if remotion was succesfull
df_books_rating.isnull().sum()

,0
Id,0
Title,0
review/text,0


In [ ]:
length_3 = len(df_books_rating)
percentage_3 = round((length_3/length_1)*100,2)

print(f"New length is: {length_3}")
print(f"Rows removed: {length_2 - length_3}")
print(f"Percentage remaining vs original dataset: {percentage_3}%")

New length is: 2619132
Rows removed: 216
Percentage remaining vs original dataset: 87.3%


# Variable for dataframe to use
Define wether to use the **whole dataset or a sample of it**. For this put:


*   **data_sample = True** --> if you want to use a sample of the original dataset.
*  **data_sample = False** --> if you want to use a the original dataset.

In [ ]:
# Variable to define if we use whole dataset or a sample (if True, we use a sample, if not the whole data)
data_sample = True

In [ ]:
# Final variable to get the data to work with
df_books_rating = df_books_rating if not data_sample else df_books_rating.sample(frac=0.05, random_state=42)

# Spark Pipeline
In this part I'll transform the reviews into final tokens to use for calculation and finding similar items.

In [ ]:
# Initialize spark
spark = sparknlp.start()

In [ ]:
# Transform pd.DataFrame into a spark.DataFrame
df_books_rating_spark = spark.createDataFrame(df_books_rating)

In [ ]:
def text_pipeline(df, input_col = 'review/text'):
  """
    Function for processing reviews and transforming them into final tokens for finding similar items.

    Parameters:
        df (DataFrame): Spark DataFrame containing the initial column input (reviews).
        input_col (str): Column containing the book's reviews.

    Returns:
        Spark DataFrame with the new column 'final_tokens', containing the cleaned and lemmatized tokens.
    """

  # Text to NLP documents
  document_assembler = DocumentAssembler().setInputCol(input_col).setOutputCol("document")

  # Tokenization
  tokenizer = Tokenizer().setInputCols(["document"]).setOutputCol("token")

  # Normalize: lower case and remove punctuations
  normalizer = Normalizer().setInputCols(["token"]).setOutputCol("normalized").setLowercase(True).setCleanupPatterns(["[^a-zA-Z0-9]"])

  # Remove stopwords
  stopwords_cleaner = StopWordsCleaner().setInputCols(["normalized"]).setOutputCol("cleanTokens").setCaseSensitive(False)

  # Lemmatization
  lemmatizer = LemmatizerModel.pretrained().setInputCols(["cleanTokens"]).setOutputCol("lemma")

  # Pipeline's end: tokens into legible format
  finisher = Finisher().setInputCols(["lemma"]).setOutputCols(["final_tokens"]).setCleanAnnotations(True)

  # Create the pipeline
  pipeline = Pipeline(stages=[document_assembler, tokenizer, normalizer, stopwords_cleaner, lemmatizer, finisher])

  # Apply pipeline to df_books_rating
  pipeline_model = pipeline.fit(df)

  # Final result
  pipeline_result = pipeline_model.transform(df)

  return pipeline_result

In [ ]:
# Create a new variable containing the final Spark DataFrame
pipeline_result = text_pipeline(df_books_rating_spark)

lemma_antbnc download started this may take some time.
Approximate size to download 907.6 KB
[OK!]


In [ ]:
# Show pipeline results with the original review column and the final column with the tokens
pipeline_result.show(10)

+----------+--------------------+--------------------+--------------------+
|        Id|               Title|         review/text|        final_tokens|
+----------+--------------------+--------------------+--------------------+
|B000NTMPGK|Violets Are Blue ...|Of all the James ...|[james, patterson...|
|0375828095|Junie B., First G...|Another story in ...|[another, story, ...|
|0809123320|The Cloud of Unkn...|Again, I seem to ...|[seem, purchase, ...|
|073933252X|Through a Glass D...|I read this book ...|[read, book, firs...|
|B000GK17RY|                 Boy|This was a great ...|[great, book, sha...|
|0972847227| It's All About Baby|Just received min...|[receive, i, orig...|
|1841010162|ON THE WAY TO BET...|Great book! Very ...|[great, book, pro...|
|1565791541|A North Carolina ...|The photography w...|[photography, lov...|
|0061065560|Wing Commander Ju...|Before I ever kne...|[ever, know, book...|
|B000OVGENM|         Cobra Event|Excellent book! D...|[excellent, book,...|
+----------+

# Shingles

In [ ]:
# Remove cases with tokens <3, as it will be the number used for shingle
filtered_pipeline_result = pipeline_result.filter(size(col("final_tokens")) >= 3)

In [ ]:
ngram = NGram(n=3, inputCol="final_tokens", outputCol="shingles")

In [ ]:
shingled_df = ngram.transform(filtered_pipeline_result)

In [ ]:
shingled_df.show(10)

+----------+--------------------+--------------------+--------------------+--------------------+
|        Id|               Title|         review/text|        final_tokens|            shingles|
+----------+--------------------+--------------------+--------------------+--------------------+
|B000NTMPGK|Violets Are Blue ...|Of all the James ...|[james, patterson...|[james patterson ...|
|0375828095|Junie B., First G...|Another story in ...|[another, story, ...|[another story se...|
|0809123320|The Cloud of Unkn...|Again, I seem to ...|[seem, purchase, ...|[seem purchase bo...|
|073933252X|Through a Glass D...|I read this book ...|[read, book, firs...|[read book first,...|
|B000GK17RY|                 Boy|This was a great ...|[great, book, sha...|[great book share...|
|0972847227| It's All About Baby|Just received min...|[receive, i, orig...|[receive i origin...|
|1841010162|ON THE WAY TO BET...|Great book! Very ...|[great, book, pro...|[great book provo...|
|1565791541|A North Carolina .

# Hashing the Shingles

In [ ]:
hashing_tf = HashingTF(inputCol="shingles", outputCol="hashed_shingles", numFeatures=2048)

In [ ]:
hashed_shingles_df = hashing_tf.transform(shingled_df)

In [ ]:
hashed_shingles_df.show(10)

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+
|        Id|               Title|         review/text|        final_tokens|            shingles|     hashed_shingles|
+----------+--------------------+--------------------+--------------------+--------------------+--------------------+
|B000NTMPGK|Violets Are Blue ...|Of all the James ...|[james, patterson...|[james patterson ...|(2048,[133,206,24...|
|0375828095|Junie B., First G...|Another story in ...|[another, story, ...|[another story se...|(2048,[62,152,163...|
|0809123320|The Cloud of Unkn...|Again, I seem to ...|[seem, purchase, ...|[seem purchase bo...|(2048,[31,42,71,9...|
|073933252X|Through a Glass D...|I read this book ...|[read, book, firs...|[read book first,...|(2048,[39,61,64,1...|
|B000GK17RY|                 Boy|This was a great ...|[great, book, sha...|[great book share...|(2048,[171,448,60...|
|0972847227| It's All About Baby|Just received min...|[r

# MinHashing

In [ ]:
minhash = MinHashLSH(inputCol="hashed_shingles", outputCol="minhashes", numHashTables=4)

In [ ]:
minhash_model = minhash.fit(hashed_shingles_df)

In [ ]:
minhashed_df = minhash_model.transform(hashed_shingles_df).cache()

In [ ]:
minhashed_df.show(10)

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|        Id|               Title|         review/text|        final_tokens|            shingles|     hashed_shingles|           minhashes|
+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|B000NTMPGK|Violets Are Blue ...|Of all the James ...|[james, patterson...|[james patterson ...|(2048,[133,206,24...|[[3.6290422E7], [...|
|0375828095|Junie B., First G...|Another story in ...|[another, story, ...|[another story se...|(2048,[62,152,163...|[[4.8992447E7], [...|
|0809123320|The Cloud of Unkn...|Again, I seem to ...|[seem, purchase, ...|[seem purchase bo...|(2048,[31,42,71,9...|[[1.9594551E7], [...|
|073933252X|Through a Glass D...|I read this book ...|[read, book, firs...|[read book first,...|(2048,[39,61,64,1...|[[1.0339328E7], [...|
|B000GK17RY|               

# Jaccard Similarity

In [ ]:
similar_pairs = minhash_model.approxSimilarityJoin(minhashed_df, minhashed_df, threshold=0.2, distCol="jaccard_distance").filter(
    (col("datasetA.id") < col("datasetB.id")) & (col("datasetA.review/text") != col("datasetB.review/text")))

In [ ]:
final_similar_pairs = similar_pairs.select(
    col("datasetA.id").alias("doc_id"),
    col("datasetA.review/text"),
    col("datasetB.id").alias("most_similar_doc_id"),
    col("datasetB.review/text"),
    col("jaccard_distance"))

In [ ]:
final_similar_pairs.show(10)

+----------+--------------------+-------------------+--------------------+-------------------+
|    doc_id|         review/text|most_similar_doc_id|         review/text|   jaccard_distance|
+----------+--------------------+-------------------+--------------------+-------------------+
|0708982581|Emma Woodhouse, "...|         8437615607|Emma Woodhouse, "...|0.15555555555555556|
|0395051150|Emma Woodhouse, "...|         8437615607|Emma Woodhouse, "...|0.15555555555555556|
|0340283947|The BLTC Kindle e...|         1582343632|This Kindle editi...|0.10580204778156999|
|B0000YSH5G|this is the BEST ...|         B000GQK706|The best book I h...|                0.0|
|0904724719|This Kindle editi...|         1582343632|This Kindle editi...|0.06028368794326244|
|B000BI4160|"The Lord of the ...|         B000H7EO2G|"The Lord of the ...|                0.0|
|B0006AL5RG|I just wanted to ...|         B000J6DLBU|I just wanted to ...|0.03546099290780147|
|0945466307|gave it as a gift...|         14000479